# Deepfake Detector - Exploration and Demo

This notebook demonstrates the usage of the deepfake detection tool with examples and visualizations.

## Setup and Imports

In [ ]:
import sys
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path().absolute().parent))

from src.face_extractor import FaceExtractor
from src.preprocessing import ImagePreprocessor
from src.model import create_model
from src.predict import DeepfakePredictor
from src.video_analyzer import VideoAnalyzer
from src.visualization import visualize_grad_cam, plot_confusion_matrix
from src.utils import load_config

print("✓ Imports successful")

## 1. Face Detection Demo

In [ ]:
# Create a sample image with face-like pattern
def create_sample_face(size=480):
    """Create a sample image with a face-like pattern."""
    image = np.random.randint(100, 150, (size, size, 3), dtype=np.uint8)
    
    # Draw face
    center_x, center_y = size // 2, size // 2
    cv2.ellipse(image, (center_x, center_y), (80, 100), 0, 0, 360, (200, 180, 160), -1)
    
    # Eyes
    cv2.circle(image, (center_x - 30, center_y - 20), 12, (50, 50, 50), -1)
    cv2.circle(image, (center_x + 30, center_y - 20), 12, (50, 50, 50), -1)
    
    # Mouth
    cv2.ellipse(image, (center_x, center_y + 40), (35, 18), 0, 0, 180, (100, 50, 50), -1)
    
    return image

# Create and display sample
sample_image = create_sample_face()
plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(sample_image, cv2.COLOR_BGR2RGB))
plt.title("Sample Image")
plt.axis('off')
plt.show()

# Extract faces
extractor = FaceExtractor(method="mediapipe")
faces = extractor.extract_faces(sample_image)

print(f"✓ Detected {len(faces)} face(s)")

# Display extracted faces
if faces:
    for idx, face_data in enumerate(faces):
        face_img = face_data['face']
        confidence = face_data['confidence']
        
        plt.figure(figsize=(6, 6))
        plt.imshow(cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB))
        plt.title(f"Extracted Face {idx+1} (Confidence: {confidence:.2f})")
        plt.axis('off')
        plt.show()

## 2. Preprocessing Demo

In [ ]:
# Create preprocessor
preprocessor = ImagePreprocessor(
    target_size=(299, 299),
    normalization="standard"
)

# Preprocess image
if faces:
    face_img = faces[0]['face']
    processed = preprocessor.preprocess_image(face_img)
    
    # Denormalize for visualization
    denormalized = preprocessor.denormalize(processed)
    
    # Display
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    axes[0].imshow(cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original Face")
    axes[0].axis('off')
    
    axes[1].imshow(denormalized)
    axes[1].set_title("Preprocessed (299x299)")
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"✓ Preprocessed shape: {processed.shape}")
    print(f"✓ Value range: [{processed.min():.2f}, {processed.max():.2f}]")

## 3. Model Architecture

In [ ]:
# Create model (without pretrained weights for demo)
model = create_model(
    input_shape=(299, 299, 3),
    architecture="xception",
    pretrained_weights=None,  # Use None for quick demo
    dropout_rate=0.5,
    dense_units=512
)

# Display model summary
print("\n" + "="*60)
print("MODEL ARCHITECTURE")
print("="*60)
model.summary()

# Count parameters
trainable, non_trainable = model.count_parameters()
print(f"\n✓ Trainable parameters: {trainable:,}")
print(f"✓ Non-trainable parameters: {non_trainable:,}")
print(f"✓ Total parameters: {trainable + non_trainable:,}")

## 4. Prediction Demo (requires trained model)

In [ ]:
# Note: This requires a trained model
# Uncomment and modify the path to use with a real trained model

# MODEL_PATH = "../models/best_model.h5"
# CONFIG_PATH = "../config/config.yaml"

# if os.path.exists(MODEL_PATH):
#     # Create predictor
#     predictor = DeepfakePredictor(MODEL_PATH, CONFIG_PATH)
#     
#     # Predict on sample image
#     result = predictor.predict_image(sample_image)
#     
#     print("\nPrediction Results:")
#     print(f"Label: {result['label']}")
#     print(f"Confidence: {result['confidence']:.2f}%")
#     print(f"Probability: {result['probability']:.4f}")
# else:
#     print(f"Model not found at {MODEL_PATH}")
#     print("Train a model first using: python scripts/train_model.py")

print("⚠️ Prediction demo requires a trained model")
print("Train a model using: python scripts/train_model.py --data-dir ./data")

## 5. Configuration

In [ ]:
# Load and display configuration
config = load_config("../config/config.yaml")

print("\n" + "="*60)
print("CONFIGURATION")
print("="*60)

import json
print(json.dumps(config, indent=2))

## 6. Training Visualization (example)

In [ ]:
# Example training history visualization
from src.visualization import plot_training_history

# Simulated training history
history = {
    'loss': [0.6, 0.5, 0.4, 0.3, 0.25, 0.2, 0.18, 0.16, 0.15, 0.14],
    'accuracy': [0.65, 0.70, 0.75, 0.80, 0.83, 0.86, 0.88, 0.90, 0.91, 0.92],
    'val_loss': [0.65, 0.55, 0.45, 0.35, 0.30, 0.28, 0.26, 0.25, 0.24, 0.23],
    'val_accuracy': [0.60, 0.68, 0.73, 0.78, 0.81, 0.83, 0.85, 0.87, 0.88, 0.89],
}

plot_training_history(history, metrics=['loss', 'accuracy'])
print("✓ Training history visualization")

## 7. Confusion Matrix (example)

In [ ]:
# Example confusion matrix
y_true = np.array([0, 0, 1, 1, 0, 1, 0, 1, 1, 0])
y_pred = np.array([0, 0, 1, 0, 0, 1, 1, 1, 1, 0])

plot_confusion_matrix(y_true, y_pred, class_names=['Real', 'Fake'], normalize=True)
print("✓ Confusion matrix visualization")

## Summary

This notebook demonstrated:

1. ✓ Face detection and extraction
2. ✓ Image preprocessing pipeline
3. ✓ Model architecture overview
4. ✓ Configuration management
5. ✓ Visualization utilities

### Next Steps:

1. Prepare training data in `data/real` and `data/fake` directories
2. Train a model: `python scripts/train_model.py --data-dir ./data`
3. Test on videos: `python scripts/detect_video.py --video test.mp4 --model models/best_model.h5`
4. Try real-time detection: `python scripts/detect_realtime.py --model models/best_model.h5`

### Resources:

- [FaceForensics++ Dataset](https://github.com/ondyari/FaceForensics)
- [DFDC Dataset](https://www.kaggle.com/c/deepfake-detection-challenge)
- [Project Repository](https://github.com/ELMAALMI/deepfake-detector)